# Notebook 03 - Inflation analysis (no inflation in B&F)

Decomposes the oil-shock response into a price and a quantity part and computes the
Domar-weighted mean price (the only 'inflation' object in the static B&F model).

CAVEAT: the `eta_price_insensitivity` helper is intentionally a no-op for the fixed-L
B&F model (there is no eta parameter here); the real eta-variance result lives in the
main BeyondHulten package. It is kept only to document the negative finding.

In [ ]:
# --- Project setup (robust path resolution) ---
# @__DIR__ resolves to this notebook's directory; we anchor on the package root.
using LinearAlgebra, Statistics, Printf, DelimitedFiles

const NOTEBOOK_DIR = @__DIR__
const REP_DIR = joinpath(NOTEBOOK_DIR, "..")          # bf_replication/
cd(REP_DIR)
include(joinpath(REP_DIR, "src", "BFReplication.jl"))
using .BFReplication
using .BFReplication.DataLoader
using .BFReplication.BFModel
using .BFReplication.InflationAnalysis

const DATA_DIR = joinpath(REP_DIR, "..", "Replication Files", "GDP Simulatin -- 88 Sector")
const RESULTS_DIR = joinpath(REP_DIR, "data", "results")
mkpath(RESULTS_DIR)

println("Project dir : ", REP_DIR)
println("Data dir    : ", DATA_DIR)
println("Results dir : ", RESULTS_DIR)

## Baseline vs oil shock: price vs quantity

In [ ]:
A = ones(data.N); A[7] = 0.7
ps = BFParameters(A, data.Omega, data.alpha, data.beta, data.L, 0.5, 0.0001, 0.9)
sol = compute_equilibrium(ps)
pq = analyze_price_vs_quantity(sol, data, 7)
println("oil-shock log price change    = $(pq.price_change)")
println("oil-shock log quantity change = $(pq.quantity_change)")

## Inflation measures (Domar-weighted mean price)

In [ ]:
measures = compute_inflation_measures(sol, data)
println("mean log price change (Domar-weighted) = $(measures.mean_log_price)")

## eta price insensitivity + network decomposition (see caveat above)

In [ ]:
eta = eta_price_insensitivity(sol, data)
println("price CV to eta = $(eta.price_cv_to_eta)   (no eta in fixed-L B&F -> no-op)")
net = network_price_decomposition(sol, data)
println("network decomposition: ", net)